In [ ]:
import joblib
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.config import load_paths
from model.model import read_scale_data
from evaluation.flux_energy import physical_values
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xarray as xr


# ============================================================
# SPECTRA
# ============================================================

def _dm(m):
    """
    Estimate spacing in |m|.

    Works even if m contains negative values.
    """
    am = np.abs(np.asarray(m))
    order = np.argsort(am)

    dm = np.empty_like(am, dtype=float)
    dm[order] = np.gradient(am[order])

    return dm


def vertical_spectrum(E, k, l, m):
    """
    Vertical spectrum E(m).

    E: energy per mode [nk, nl, nm]

    First sum over horizontal wavenumbers (k,l),
    then divide by the vertical-wavenumber spacing dm.
    """

    am = np.abs(m)
    order = np.argsort(am)

    dm = _dm(m)

    # Sum over k and l
    Em = E.sum(axis=(0, 1))

    # Convert energy per mode -> energy density per unit m
    Em = Em / dm

    return am[order], Em[order]


def horizontal_spectrum(E, k, l, m):
    """
    Isotropic horizontal spectrum E(k_h).

    k_h = sqrt(k^2 + l^2)

    k and l may contain both positive and negative
    wavenumbers.

    E: [nk, nl, nm]
    """

    # --------------------------------------------------------
    # Sum over vertical wavenumber
    # --------------------------------------------------------
    E2d = E.sum(axis=2)   # [nk, nl]

    # --------------------------------------------------------
    # Construct all (k,l) combinations
    # --------------------------------------------------------
    K, L = np.meshgrid(k, l, indexing="ij")

    # --------------------------------------------------------
    # Horizontal wavenumber magnitude
    # --------------------------------------------------------
    kh_mode = np.sqrt(K**2 + L**2)

    # Flatten
    kh_flat = kh_mode.ravel()
    E_flat = E2d.ravel()

    # --------------------------------------------------------
    # Remove zero horizontal wavenumber
    # --------------------------------------------------------
    valid = kh_flat > 0

    kh_flat = kh_flat[valid]
    E_flat = E_flat[valid]

    # --------------------------------------------------------
    # Sort by kh
    # --------------------------------------------------------
    order = np.argsort(kh_flat)

    kh_flat = kh_flat[order]
    E_flat = E_flat[order]

    # --------------------------------------------------------
    # Group modes with identical kh
    #
    # Use a tolerance because of floating-point arithmetic.
    # --------------------------------------------------------
    kh_unique = []
    E_shell = []

    tolerance = 1e-10

    current_kh = kh_flat[0]
    current_E = 0.0

    for kh_i, E_i in zip(kh_flat, E_flat):

        if np.isclose(
            kh_i,
            current_kh,
            rtol=tolerance,
            atol=tolerance
        ):
            current_E += E_i

        else:
            kh_unique.append(current_kh)
            E_shell.append(current_E)

            current_kh = kh_i
            current_E = E_i

    # Add final shell
    kh_unique.append(current_kh)
    E_shell.append(current_E)

    kh = np.asarray(kh_unique)
    E_shell = np.asarray(E_shell)

    # --------------------------------------------------------
    # Convert shell-integrated energy to E(k_h)
    #
    # Use midpoint shell widths.
    # --------------------------------------------------------
    if len(kh) > 1:

        edges = np.empty(len(kh) + 1)

        edges[1:-1] = 0.5 * (
            kh[:-1] + kh[1:]
        )

        # Extrapolate outer boundaries
        edges[0] = kh[0] - 0.5 * (
            kh[1] - kh[0]
        )

        edges[-1] = kh[-1] + 0.5 * (
            kh[-1] - kh[-2]
        )

        dkh = np.diff(edges)

    else:
        dkh = np.ones_like(kh)

    Ekh = E_shell / dkh

    return kh, Ekh


# ============================================================
# POWER-LAW FIT
# ============================================================

def fit_slope(x, y, mask=None):
    """
    Fit y = C x^s in log-log space.
    """

    valid = (
        np.isfinite(x) &
        np.isfinite(y) &
        (x > 0) &
        (y > 0)
    )

    if mask is not None:
        valid &= mask

    if valid.sum() < 2:
        return np.nan, np.nan

    p = np.polyfit(
        np.log10(x[valid]),
        np.log10(y[valid]),
        1
    )

    slope = p[0]
    coefficient = 10 ** p[1]

    return slope, coefficient


# ============================================================
# REFERENCE POWER LAWS
# ============================================================

def add_ref(ax, x, y, slope, label, ls, offset=0.4):

    valid = (
        np.isfinite(x) &
        np.isfinite(y) &
        (x > 0) &
        (y > 0)
    )

    if valid.sum() < 2:
        return

    x_valid = x[valid]
    y_valid = y[valid]

    # Geometric center
    x0 = np.sqrt(
        x_valid.min() *
        x_valid.max()
    )

    # Interpolate y at x0
    y0 = (
        10 **
        np.interp(
            np.log10(x0),
            np.log10(x_valid),
            np.log10(y_valid)
        )
    )

    y0 *= 10 ** (-offset)

    xs = np.array([
        x_valid.min(),
        x_valid.max()
    ])

    ax.loglog(
        xs,
        y0 * (xs / x0) ** slope,
        ls,
        color="0.4",
        lw=1.1,
        label=label
    )


# ============================================================
# WAVELENGTH CONVERSION
# ============================================================

def m2lam(x):
    """
    Convert wavenumber [rad/m] to wavelength [km]:

        lambda = 2*pi / k

    """
    x = np.asarray(x, dtype=float)

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):
        return np.where(
            x > 0,
            2 * np.pi / x / 1000,
            np.nan
        )


# ============================================================
# PLOT PANEL
# ============================================================

def panel(
    ax,
    x,
    y,
    color,
    marker,
    refs,
    fit_label,
    xlabel,
    ylabel,
    xticks,
    xtick_scale,
    top_ticks_km,
    top_label,
    fit_mask=None,
    fs=11
):

    # --------------------------------------------------------
    # Fit
    # --------------------------------------------------------
    s, c = fit_slope(
        x,
        y,
        fit_mask
    )

    # Valid points
    sel = (
        np.isfinite(x) &
        np.isfinite(y) &
        (x > 0) &
        (y > 0)
    )

    if fit_mask is not None:
        sel &= fit_mask

    # --------------------------------------------------------
    # Plot fitted power law
    # --------------------------------------------------------
    if np.isfinite(s) and np.isfinite(c) and sel.sum() >= 2:

        xs = np.array([
            x[sel].min(),
            x[sel].max()
        ])

        ax.loglog(
            xs,
            c * xs ** s,
            "-",
            color=color,
            alpha=0.3,
            lw=6,
            label=rf"{fit_label}$^{{{s:.2f}}}$"
        )

    else:

        print(
            f"WARNING: Could not fit {fit_label}. "
            f"Only {sel.sum()} valid points."
        )

    # --------------------------------------------------------
    # Spectrum
    # --------------------------------------------------------
    valid_plot = (
        np.isfinite(x) &
        np.isfinite(y) &
        (x > 0) &
        (y > 0)
    )

    ax.loglog(
        x[valid_plot],
        y[valid_plot],
        marker + "-",
        color=color,
        ms=6,
        lw=1.6
    )

    # --------------------------------------------------------
    # Reference slopes
    # --------------------------------------------------------
    for slope, lab, ls in refs:

        add_ref(
            ax,
            x,
            y,
            slope,
            lab,
            ls
        )

    # --------------------------------------------------------
    # Limits
    # --------------------------------------------------------
    xmin = x[valid_plot].min()
    xmax = x[valid_plot].max()

    ax.set_xlim(
        xmin * 0.8,
        xmax * 1.25
    )

    # --------------------------------------------------------
    # x ticks
    # --------------------------------------------------------
    xticks_valid = xticks[
        (xticks >= xmin * 0.9) &
        (xticks <= xmax * 1.1)
    ]

    ax.set_xticks(xticks_valid)

    ax.set_xticklabels(
        [
            f"{v * xtick_scale:.1f}"
            for v in xticks_valid
        ]
    )

    ax.xaxis.set_minor_formatter(
        mticker.NullFormatter()
    )

    ax.yaxis.set_minor_formatter(
        mticker.NullFormatter()
    )

    # --------------------------------------------------------
    # labels
    # --------------------------------------------------------
    ax.set_xlabel(
        xlabel,
        fontsize=fs
    )

    ax.set_ylabel(
        ylabel,
        fontsize=fs
    )

    # --------------------------------------------------------
    # Secondary wavelength axis
    # --------------------------------------------------------
    sec = ax.secondary_xaxis(
        "top",
        functions=(m2lam, m2lam)
    )

    sec.set_xticks(
        top_ticks_km
    )

    sec.set_xticklabels(
        [
            f"{v:g}"
            for v in top_ticks_km
        ]
    )

    sec.xaxis.set_minor_locator(
        mticker.NullLocator()
    )

    sec.set_xlabel(
        top_label,
        fontsize=fs
    )

    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------
    ax.grid(
        True,
        which="major",
        alpha=0.25
    )

    ax.tick_params(
        labelsize=fs - 1
    )

    ax.legend(
        fontsize=fs - 2,
        frameon=False,
        loc="lower left"
    )

    return s


# ============================================================
# LOAD DATA
# ============================================================

paths = load_paths()

save_dir = paths["save_dir"]


(
    _,
    _,
    y_train_ocean,
    _,
    _,
    _,
    _,
    ntri_ocean,
    _,
    _,
    _,
    omega_hat_train_ocean,
    _,
    _,
    _,
    _,
    _
) = read_scale_data(
    model_save_dir=save_dir,
    coarse_action_dir_jan=paths["coarse_action_dir_jan"],
    land_ocean_tag="ocean",
)


# ============================================================
# RESHAPE
# ============================================================

N_train_ocean = (
    np.expm1(y_train_ocean)
    .reshape(-1, 6, 6, 4)
)

omega_hat_train_ocean = (
    omega_hat_train_ocean
    .reshape(-1, 6, 6, 4)
)


# ============================================================
# ENERGY
# ============================================================

E_train_ocean = (
    N_train_ocean *
    omega_hat_train_ocean
)

# Mean over training samples
E_mean_train_ocean = np.mean(
    E_train_ocean,
    axis=0
)


# ============================================================
# READ WAVENUMBERS
# ============================================================

with xr.open_dataset(
    paths["coarse_action_dir_jan"]
    + "coarse_gw_spectrum_group_1_fileslide_0.nc"
) as ds:

    k_lunch = ds.k.data
    l_lunch = ds.l.data

    # Your convention
    m_lunch = -ds.m.data


# ============================================================
# USE MEAN ENERGY
# ============================================================

E = E_mean_train_ocean


# ============================================================
# COMPUTE SPECTRA
# ============================================================

mv, Ev = vertical_spectrum(
    E,
    k_lunch,
    l_lunch,
    m_lunch
)

kh, Eh = horizontal_spectrum(
    E,
    k_lunch,
    l_lunch,
    m_lunch
)


# ============================================================
# DEBUG OUTPUT
# ============================================================

print("\n==============================================")
print("WAVENUMBERS")
print("==============================================")

print("k =", k_lunch)
print("l =", l_lunch)
print("m =", m_lunch)

print("\n==============================================")
print("HORIZONTAL SPECTRUM")
print("==============================================")

print("kh [rad/m] =")
print(kh)

print("\nkh [1e-4 rad/m] =")
print(kh * 1e4)

print("\nEh =")
print(Eh)

print("\nfinite kh:", np.all(np.isfinite(kh)))
print("finite Eh:", np.all(np.isfinite(Eh)))
print("positive kh:", np.all(kh > 0))
print("positive Eh:", np.all(Eh > 0))

print("\n==============================================")
print("VERTICAL SPECTRUM")
print("==============================================")

print("mv [rad/m] =")
print(mv)

print("Ev =")
print(Ev)


# ============================================================
# FIT RANGE
# ============================================================

fit_mmin = None

if fit_mmin is None:
    vmask = None
else:
    vmask = mv >= fit_mmin


# ============================================================
# HORIZONTAL TICKS
# ============================================================

cand = np.array([
    0.3,
    0.5,
    1.0,
    1.5,
    2.0
]) * 1e-4

kh_ticks = cand[
    (cand >= kh.min() * 0.9) &
    (cand <= kh.max() * 1.1)
]


# ============================================================
# WAVELENGTH TICKS
# ============================================================

lam_range = m2lam(
    np.array([
        kh.max(),
        kh.min()
    ])
)

cand_km = np.array([
    200,
    150,
    100,
    70,
    50,
    40,
    30
])

lam_ticks = cand_km[
    (cand_km >= lam_range[0]) &
    (cand_km <= lam_range[1])
]


# ============================================================
# PLOT
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(6.8, 3.1),
    constrained_layout=True
)


# ------------------------------------------------------------
# Vertical spectrum
# ------------------------------------------------------------

sv = panel(
    axes[0],
    mv,
    Ev,
    "tab:blue",
    "o",
    refs=[
        (-3, r"$m^{-3}$", "--"),
        (-2, r"$m^{-2}$", ":")
    ],
    fit_label=r"fit: $m$",
    xlabel=r"$|m|\ \ [10^{-3}\ \mathrm{rad\,m^{-1}}]$",
    ylabel=r"$E(m)$",
    xticks=mv,
    xtick_scale=1e3,
    top_ticks_km=[
        12,
        7,
        3.5,
        1.5
    ],
    top_label=r"$\lambda_z\ \ [\mathrm{km}]$",
    fit_mask=vmask
)


# ------------------------------------------------------------
# Horizontal spectrum
# ------------------------------------------------------------

sh = panel(
    axes[1],
    kh,
    Eh,
    "tab:orange",
    "s",
    refs=[
        (-5/3, r"$k_h^{-5/3}$", "--"),
        (-3, r"$k_h^{-3}$", ":")
    ],
    fit_label=r"fit: $k_h$",
    xlabel=r"$k_h\ \ [10^{-4}\ \mathrm{rad\,m^{-1}}]$",
    ylabel=r"$E(k_h)$",
    xticks=kh_ticks,
    xtick_scale=1e4,
    top_ticks_km=lam_ticks,
    top_label=r"$\lambda_h\ \ [\mathrm{km}]$"
)


# ============================================================
# SAVE
# ============================================================

out = "/work/bb1097/b383521/plots/paper_plots"

plt.savefig(
    f"{out}/power_law_spectra.png",
    dpi=400,
    bbox_inches="tight"
)

plt.savefig(
    f"{out}/power_law_spectra.pdf",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n==============================================")
print("POWER-LAW SLOPES")
print("==============================================")

print(
    f"vertical slope   = {sv:.2f}"
)

print(
    f"horizontal slope = {sh:.2f}"
)

print(
    "\nkh shells [1e-4 rad/m]:",
    np.round(kh * 1e4, 3)
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xarray as xr


# ============================================================
# SPECTRA
# ============================================================

def _dm(m):
    """Estimate spacing in |m| (works with negative m)."""
    am = np.abs(np.asarray(m))
    order = np.argsort(am)
    dm = np.empty_like(am, dtype=float)
    dm[order] = np.gradient(am[order])
    return dm


def vertical_spectrum(E, k, l, m):
    """
    Vertical energy spectrum E(m).
    E: energy per mode [nk, nl, nm]
    Sum over (k,l), divide by the m-bin width.
    (Assumes E is energy per bin. If your m bins are log-spaced,
    this division adds -1 to the slope.)
    """
    am = np.abs(m)
    order = np.argsort(am)
    Em = E.sum(axis=(0, 1)) / _dm(m)
    return am[order], Em[order]


def horizontal_spectrum(E, k, l, m):
    """
    Isotropic horizontal energy spectrum E(k_h), k_h = sqrt(k^2 + l^2).
    Modes with identical k_h are grouped into shells.
    Returns kh, E(kh), number of modes per shell.
    """
    E2d = E.sum(axis=2)                          # [nk, nl]
    K, L = np.meshgrid(k, l, indexing="ij")
    kh_flat = np.sqrt(K**2 + L**2).ravel()
    E_flat = E2d.ravel()

    valid = kh_flat > 0
    kh_flat, E_flat = kh_flat[valid], E_flat[valid]

    order = np.argsort(kh_flat)
    kh_flat, E_flat = kh_flat[order], E_flat[order]

    kh_unique, E_shell, n_modes = [], [], []
    cur_kh, cur_E, cur_n = kh_flat[0], 0.0, 0

    for kh_i, E_i in zip(kh_flat, E_flat):
        if np.isclose(kh_i, cur_kh, rtol=1e-10, atol=1e-10):
            cur_E += E_i
            cur_n += 1
        else:
            kh_unique.append(cur_kh)
            E_shell.append(cur_E)
            n_modes.append(cur_n)
            cur_kh, cur_E, cur_n = kh_i, E_i, 1

    kh_unique.append(cur_kh)
    E_shell.append(cur_E)
    n_modes.append(cur_n)

    kh = np.asarray(kh_unique)
    E_shell = np.asarray(E_shell)
    n_modes = np.asarray(n_modes)

    # shell widths from midpoints
    if len(kh) > 1:
        edges = np.empty(len(kh) + 1)
        edges[1:-1] = 0.5 * (kh[:-1] + kh[1:])
        edges[0] = kh[0] - 0.5 * (kh[1] - kh[0])
        edges[-1] = kh[-1] + 0.5 * (kh[-1] - kh[-2])
        dkh = np.diff(edges)
    else:
        dkh = np.ones_like(kh)

    return kh, E_shell / dkh, n_modes


# ============================================================
# POWER-LAW FIT
# ============================================================

def fit_slope(x, y, mask=None):
    """Fit y = C x^s in log-log space."""
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if mask is not None:
        valid &= mask
    if valid.sum() < 2:
        return np.nan, np.nan
    p = np.polyfit(np.log10(x[valid]), np.log10(y[valid]), 1)
    return p[0], 10 ** p[1]


# ============================================================
# REFERENCE POWER LAWS
# ============================================================

def add_ref(ax, x, y, slope, label, ls, offset=0.4):
    valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if valid.sum() < 2:
        return

    xv, yv = x[valid], y[valid]
    x0 = np.sqrt(xv.min() * xv.max())
    y0 = 10 ** np.interp(np.log10(x0), np.log10(xv), np.log10(yv))
    y0 *= 10 ** (-offset)

    xs = np.array([xv.min(), xv.max()])
    ax.loglog(xs, y0 * (xs / x0) ** slope, ls, color="0.4", lw=1.1, label=label)


# ============================================================
# WAVELENGTH CONVERSION
# ============================================================

def m2lam(x):
    """Wavenumber [rad/m] -> wavelength [km]."""
    x = np.asarray(x, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(x > 0, 2 * np.pi / x / 1000, np.nan)


# ============================================================
# PLOT PANEL
# ============================================================

def panel(ax, x, y, color, marker, refs, xlabel, ylabel, xticks, xtick_scale,
          top_ticks_km, top_label, fit=True, fit_label="", slope_err=None,
          fit_mask=None, fs=11):

    s = np.nan

    # ---------------- optional fit ----------------
    if fit:
        s, c = fit_slope(x, y, fit_mask)

        sel = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        if fit_mask is not None:
            sel &= fit_mask

        if np.isfinite(s) and np.isfinite(c) and sel.sum() >= 2:
            xs = np.array([x[sel].min(), x[sel].max()])
            if slope_err is not None and np.isfinite(slope_err):
                lab = rf"{fit_label}$^{{{s:.2f}\pm{slope_err:.2f}}}$"
            else:
                lab = rf"{fit_label}$^{{{s:.2f}}}$"
            ax.loglog(xs, c * xs ** s, "-", color=color, alpha=0.3, lw=6,
                      label=lab)
        else:
            print(f"WARNING: could not fit {fit_label}.")

    # ---------------- spectrum ----------------
    vp = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    ax.loglog(x[vp], y[vp], marker + "-", color=color, ms=6, lw=1.6)

    # ---------------- reference slopes ----------------
    for slope, lab, ls in refs:
        add_ref(ax, x, y, slope, lab, ls)

    # ---------------- limits and ticks ----------------
    xmin, xmax = x[vp].min(), x[vp].max()
    ax.set_xlim(xmin * 0.8, xmax * 1.25)

    xt = xticks[(xticks >= xmin * 0.9) & (xticks <= xmax * 1.1)]
    ax.set_xticks(xt)
    ax.set_xticklabels([f"{v * xtick_scale:.1f}" for v in xt])
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    ax.yaxis.set_minor_formatter(mticker.NullFormatter())

    ax.set_xlabel(xlabel, fontsize=fs)
    ax.set_ylabel(ylabel, fontsize=fs)

    # ---------------- secondary wavelength axis ----------------
    sec = ax.secondary_xaxis("top", functions=(m2lam, m2lam))
    sec.set_xticks(top_ticks_km)
    sec.set_xticklabels([f"{v:g}" for v in top_ticks_km])
    sec.xaxis.set_minor_locator(mticker.NullLocator())
    sec.set_xlabel(top_label, fontsize=fs)

    ax.grid(True, which="major", alpha=0.25)
    ax.tick_params(labelsize=fs - 1)
    ax.legend(fontsize=fs - 2, frameon=False, loc="lower left")

    return s


# ============================================================
# LOAD DATA  (uses your own helpers)
# ============================================================

paths = load_paths()
save_dir = paths["save_dir"]

(
    _, _, y_train_ocean, _, _, _, _, ntri_ocean, _, _, _,
    omega_hat_train_ocean, _, _, _, _, _
) = read_scale_data(
    model_save_dir=save_dir,
    coarse_action_dir_jan=paths["coarse_action_dir_jan"],
    land_ocean_tag="ocean",
)


# ============================================================
# RESHAPE + ENERGY
# ============================================================

N_train_ocean = np.expm1(y_train_ocean).reshape(-1, 6, 6, 4)
omega_hat_train_ocean = omega_hat_train_ocean.reshape(-1, 6, 6, 4)

# Energy = wave action * intrinsic frequency
E_train_ocean = N_train_ocean * omega_hat_train_ocean        # [n, 6, 6, 4]
E = E_train_ocean.mean(axis=0)                               # [6, 6, 4]


# ============================================================
# READ WAVENUMBERS
# ============================================================

with xr.open_dataset(
    paths["coarse_action_dir_jan"] + "coarse_gw_spectrum_group_1_fileslide_0.nc"
) as ds:
    k_lunch = ds.k.data
    l_lunch = ds.l.data
    m_lunch = -ds.m.data


# ============================================================
# SPECTRA
# ============================================================

mv, Ev = vertical_spectrum(E, k_lunch, l_lunch, m_lunch)
kh, Eh, n_modes = horizontal_spectrum(E, k_lunch, l_lunch, m_lunch)


# ============================================================
# DIAGNOSTICS (check the m normalization!)
# ============================================================

order_m = np.argsort(np.abs(m_lunch))
Em_raw = E.sum(axis=(0, 1))[order_m]

print("\n=== VERTICAL ===")
print("|m| [rad/m]        :", mv)
print("bin widths dm       :", _dm(m_lunch)[order_m])
print("dm/|m| (const => log-spaced bins):", _dm(m_lunch)[order_m] / mv)
print("E per bin (raw)     :", Em_raw)
print("E(m) (per unit m)   :", Ev)
print("slope raw per-bin   :", fit_slope(mv, Em_raw)[0])
print("slope density E(m)  :", fit_slope(mv, Ev)[0], "(reference only)")

print("\n=== HORIZONTAL ===")
print("kh [1e-4 rad/m]     :", np.round(kh * 1e4, 3))
print("modes per shell     :", n_modes)
print("E(kh)               :", Eh)


# ============================================================
# SLOPES
# ============================================================

sh_fit = fit_slope(kh, Eh)[0]

print("\n=== SLOPES ===")
print(f"horizontal slope = {sh_fit:.2f}")
print(f"vertical slope   = {fit_slope(mv, Ev)[0]:.2f}  (4 points, not shown in figure)")


# ============================================================
# TICKS
# ============================================================

cand = np.array([0.3, 0.5, 1.0, 1.5, 2.0]) * 1e-4
kh_ticks = cand[(cand >= kh.min() * 0.9) & (cand <= kh.max() * 1.1)]

lam_range = m2lam(np.array([kh.max(), kh.min()]))
cand_km = np.array([200, 150, 100, 70, 50, 40, 30])
lam_ticks = cand_km[(cand_km >= lam_range[0]) & (cand_km <= lam_range[1])]


# ============================================================
# PLOT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(6.8, 3.1), constrained_layout=True)

# Vertical: NO fit, only reference lines (4 coarse bins, mostly below m*)
panel(
    axes[0], mv, Ev, "tab:blue", "o",
    refs=[(-3, r"$m^{-3}$", "--"), (-2, r"$m^{-2}$", ":")],
    xlabel=r"$|m|\ \ [10^{-3}\ \mathrm{rad\,m^{-1}}]$",
    ylabel=r"$E(m)$",
    xticks=mv,
    xtick_scale=1e3,
    top_ticks_km=[12, 7, 3.5, 1.5],
    top_label=r"$\lambda_z\ \ [\mathrm{km}]$",
    fit=False,
)

# Horizontal: fit
panel(
    axes[1], kh, Eh, "tab:orange", "s",
    refs=[(-5/3, r"$k_h^{-5/3}$", "--"), (-3, r"$k_h^{-3}$", ":")],
    xlabel=r"$k_h\ \ [10^{-4}\ \mathrm{rad\,m^{-1}}]$",
    ylabel=r"$E(k_h)$",
    xticks=kh_ticks,
    xtick_scale=1e4,
    top_ticks_km=lam_ticks,
    top_label=r"$\lambda_h\ \ [\mathrm{km}]$",
    fit=True,
    fit_label=r"fit: $k_h$",
)


# ============================================================
# SAVE
# ============================================================

out = "/work/bb1097/b383521/plots/paper_plots"
os.makedirs(out, exist_ok=True)

plt.savefig(f"{out}/power_law_spectra.png", dpi=400, bbox_inches="tight")
plt.savefig(f"{out}/power_law_spectra.pdf", bbox_inches="tight")
plt.show()